# Huấn luyện Faster R-CNN cho nhận diện biển báo (Zalo AI 2020)
Notebook này được thiết kế để chạy trên **Google Colab** (GPU T4). Nó sử dụng PyTorch thuần để xây dựng DataLoader và vòng lặp huấn luyện.

In [ ]:
# Tải bộ dữ liệu Zalo AI từ Kaggle về Google Colab bằng API (Yêu cầu phải upload file kaggle.json lên Colab trước)
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d phhasian0710/za-traffic-2020
!unzip -q -n za-traffic-2020.zip -d /content/dataset

In [ ]:
# Tải thư viện cần thiết
import os
import json
import torch
import torch.utils.data
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

# Kiểm tra xem GPU có sẵn sàng không
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang sử dụng thiết bị: {device}")
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.cluster import KMeans
import numpy as np


In [ ]:
# Khởi tạo lớp Đọc Dữ liệu tích hợp Albumentations (BBox-Safe Crop)
class ZaloTrafficDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, json_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        with open(json_file, 'r', encoding='utf-8') as f:
            self.coco_data = json.load(f)
        
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.image_to_annotations = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.image_to_annotations:
                self.image_to_annotations[img_id] = []
            self.image_to_annotations[img_id].append(ann)
            
        self.image_ids = list(self.image_to_annotations.keys())

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        
        img_path = os.path.join(self.root_dir, img_info['file_name'])
        # Chuyển PIL Image thành Numpy Array cho Albumentations
        img = np.array(Image.open(img_path).convert("RGB"))
        
        annos = self.image_to_annotations.get(img_id, [])
        boxes = []
        labels = []
        
        for anno in annos:
            x_min, y_min, w, h = anno['bbox']
            x_max = x_min + w
            y_max = y_min + h
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(anno['category_id']) # 1-7
            
        if self.transform and len(boxes) > 0:
            transformed = self.transform(image=img, bboxes=boxes, labels=labels)
            img = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']
        elif self.transform:
            # Ảnh rỗng (không có biển báo)
            transformed = self.transform(image=img, bboxes=[], labels=[])
            img = transformed['image']
            
        if not isinstance(img, torch.Tensor):
            img = F.to_tensor(img)
            
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([img_id])
        
        return img, target

    def __len__(self):
        return len(self.image_ids)

# Khởi tạo chuỗi Augmentation
def get_transform():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        # An toàn cắt ảnh: Hủy cắt nếu biển báo bị mất > 50%
        A.RandomSizedBBoxSafeCrop(width=1280, height=1280, erosion_rate=0.0, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.5))



In [ ]:
# Khai báo đường dẫn dữ liệu
root_dir = '/content/dataset/za_traffic_2020/traffic_train/images'
json_file = '/content/dataset/za_traffic_2020/traffic_train/train_traffic_sign_dataset.json'

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = ZaloTrafficDataset(root_dir, json_file, transform=get_transform())
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=4, shuffle=True, collate_fn=collate_fn
)
print(f"Đã tạo DataLoader với {len(dataset)} ảnh hợp lệ.")

# [TỪ EDA E6] TỰ ĐỘNG CHẠY K-MEANS ĐỂ TÌM ANCHOR BOX
print("Đang phân tích K-Means 5 cụm cho Anchor Box từ tập dữ liệu...")
all_boxes = []
for ann in dataset.coco_data['annotations']:
    w, h = ann['bbox'][2], ann['bbox'][3]
    all_boxes.append([w, h])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(all_boxes)
centers = np.sort(kmeans.cluster_centers_, axis=0)
anchor_sizes_kmeans = tuple(int(center[0]) for center in centers)
print(f"5 Kích thước Anchor thu được: {anchor_sizes_kmeans}")

ANCHOR_SIZES = (anchor_sizes_kmeans, )
ASPECT_RATIOS = ((1.0,), ) # Mặc định Zalo AI biển báo hình vuông 1:1


In [ ]:
from torchvision.models.detection.anchor_utils import AnchorGenerator

# Hàm tạo mô hình Faster R-CNN với K-Means Anchors
def get_model(num_classes, anchor_sizes, aspect_ratios):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Ép K-Means Anchor Generator vào RPN
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model



In [ ]:
# Gọi thư viện hiển thị thanh tiến trình và kết nối Drive
from tqdm import tqdm
from google.colab import drive
import os

# Yêu cầu quyền truy cập Google Drive để lưu file vĩnh viễn
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/DoAn_NhanDienBienBao/faster_rcnn_highres'
os.makedirs(save_dir, exist_ok=True) 

# Chuẩn bị huấn luyện
num_classes = 8  # [QUAN TRỌNG] 7 class biển báo + 1 class nền (background)
model = get_model(num_classes, ANCHOR_SIZES, ASPECT_RATIOS)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
# Giữ nguyên SGD thay vì AdamW để đảm bảo ResNet hội tụ tốt nhất
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10 
print("Bắt đầu huấn luyện mô hình Faster R-CNN...")

for epoch in range(num_epochs):
    model.train()  
    epoch_loss = 0
    
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())

    print(f"\nHoàn thành Epoch {epoch+1} - Tổng độ lỗi (Loss): {epoch_loss:.4f}")
    
    save_path = os.path.join(save_dir, 'faster_rcnn_last.pth')
    torch.save(model.state_dict(), save_path)
    print(f"[Bảo mật] Đã tự động lưu checkpoint của Epoch {epoch+1} vào Drive.")

final_path = os.path.join(save_dir, 'faster_rcnn_best.pth')
torch.save(model.state_dict(), final_path)
print(f"Đã lưu mô hình vĩnh viễn và an toàn tại: {final_path}")

